<a href="https://colab.research.google.com/github/adljna/Blu-SentimentAnalysis/blob/main/Week%202-3/2-Preprocessing-BluAppReview.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Data Preprocessing Blu App Review**
This notebook continues the review scraping process previously explained in [1-scraping.ipynb](https://colab.research.google.com/drive/1XOnxtGNkffOdXslITOG4ay1UeZ5lXTui?usp=sharing)

# **Preprocessing Pipeline**

> In this stage, a series of text preprocessing steps are applied to clean and standardize the review data before further analysis. This process aims to reduce noise, improve text quality, and enhance the performance of subsequent sentiment analysis.



The preprocessing pipeline consists of the following steps:
1. Lowercasing: Converting all text into lowercase to ensure consistency.
3. Punctuation Removal: Removing punctuation marks that do not contribute to meaning.
4. Stopword Removal: Eliminating commonly used words (e.g., "and", "the") that carry little semantic value.
2. Tokenization: Splitting text into individual words or tokens.
5. Stemming/Lemmatization: Reducing words to their root or base form.
6. Expand Contractions: Converting shortened words (e.g., "don't" → "do not") into their full forms.
7. Spelling Correction: Correcting misspelled words to improve text accuracy.
8. Rare Words Removal: Removing infrequent words that may introduce noise.
9. Common Words Removal: Filtering overly frequent words that do not add meaningful information.




## **Setup and Import Libraries**



In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import regex
import re
import os
import string
import emoji
from collections import Counter

import nltk
from nltk import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

import wordcloud
from wordcloud import WordCloud

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

### **Load Dataset**

In [3]:
file_path = '../data/bluapp_reviews.csv'

if os.path.exists(file_path):
    df_bluRev = pd.read_csv(file_path)
    print("Successfully loaded CSV into df_bluRev")
    print(os.path.abspath(file_path))
else:
    print("File not found:", os.path.abspath(file_path))
    df_bluRev = pd.DataFrame()

df_bluRev.head()

Successfully loaded CSV into df_bluRev
c:\Users\Lenovo\Documents\COLLEGE\SEMESTER 6\PENGELOLAAN BAHASA ALAMI\BluApp Review\Blu-SentimentAnalysis\data\bluapp_reviews.csv


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,11a53b42-c2cd-4506-bc03-81f53ef395e3,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"verifikasi wajah tidak jelas,stuck di pastikan...",1,0,NaN,2026-05-16 23:08:41,"Hai, Kak Darma. Maaf atas ketidaknyamanannya. ...",2026-05-17 00:07:20,NaN
1,50f6878c-3db8-447e-9603-e1676aa60114,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,hihihihi kog ngeri nian baca review penggunany...,1,0,NaN,2026-05-16 22:10:32,"Hai, Kak Maulana. Maaf atas ketidaknyamanannya...",2026-05-16 23:06:54,NaN
2,dfb3aa7a-273e-40f8-8c1d-e839818b53ce,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,gak perlu di instal hanya menghabiskan paket d...,1,0,2.12.10,2026-05-16 21:52:46,"Hai, Kak Jaki. Maaf atas ketidaknyamanannya. K...",2026-05-16 21:55:55,2.12.10
3,6635a38a-b66c-4012-9bbf-7b08d99862bd,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,wish keren sih..ini bank digital yang aku suka..,5,0,2.12.10,2026-05-16 21:11:37,"Hai, Kak Alan. Terima kasih atas ulasannya. Se...",2026-05-16 21:54:10,2.12.10
4,3fa58886-c2f2-456a-8007-bc7888445931,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"aplikasi kocak, verifikasi wajah gak bisa bisa...",1,0,NaN,2026-05-16 20:47:26,"Hai, Kak Muhammad. Maaf atas ketidaknyamananny...",2026-05-16 21:53:27,NaN


In [4]:
# Display dataset structure and check data types and missing values
df_bluRev.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26530 entries, 0 to 26529
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              26530 non-null  object
 1   userName              26530 non-null  object
 2   userImage             26530 non-null  object
 3   content               26530 non-null  object
 4   score                 26530 non-null  int64 
 5   thumbsUpCount         26530 non-null  int64 
 6   reviewCreatedVersion  22173 non-null  object
 7   at                    26530 non-null  object
 8   replyContent          26530 non-null  object
 9   repliedAt             26530 non-null  object
 10  appVersion            22173 non-null  object
dtypes: int64(2), object(9)
memory usage: 2.2+ MB


### **Drop Unnecessary Column Selection**

In [5]:
# Drop reviewId, userName, userImage
df_bluRev = df_bluRev.loc[:,["content","score","thumbsUpCount", "reviewCreatedVersion", "at", "replyContent", "repliedAt"]]

### **Case Folding**

In [6]:
# Make sure the content column is of type string
df_bluRev['content'] = df_bluRev['content'].astype(str)

# Change all text to lowercase
df_bluRev['cleaned_content'] = df_bluRev['content'].str.lower()

# Example after lowercasing
print("\nAfter lowercasing:")
print(df_bluRev['cleaned_content'][1])


After lowercasing:
hihihihi kog ngeri nian baca review penggunanya. macam horror kali rasanya. ga jadi deh bikin rekening. uninstall. wassalam.


### **Punctuation & Special Characters Removal**

In [7]:
def clean_text(text):
    text = str(text)

    # Remove emoji
    text = re.sub(r'[\U00010000-\U0010ffff]', '', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove special characters (keep letters)
    text = re.sub(r'[^a-z\s]', '', text)
    return text

df_bluRev['cleaned_content'] = df_bluRev['cleaned_content'].apply(clean_text)

# Example
print(df_bluRev['cleaned_content'].iloc[1])

hihihihi kog ngeri nian baca review penggunanya macam horror kali rasanya ga jadi deh bikin rekening uninstall wassalam


### **Text Normalization - Spelling Correction**

In [8]:
# Indonesian slang dictionary based on app reviews
idslang_dict = {
    'gak': 'tidak', 'ga': 'tidak', 'gk': 'tidak', 'g': 'tidak',
    'tdk': 'tidak', 'nggak': 'tidak', 'ngga': 'tidak', 'kagak': 'tidak',
    'engga': 'tidak', 'enggak': 'tidak', 'ngk': 'tidak',
    'tp': 'tapi', 'tpi': 'tapi',
    'krn': 'karena', 'karna': 'karena', 'krna': 'karena',
    'sy': 'saya', 'aku': 'saya', 'gw': 'saya', 'gue': 'saya', 'gua': 'saya', 'w': 'saya',
    'lo': 'kamu', 'lu': 'kamu',
    'yg': 'yang', 'yng': 'yang',
    'dg': 'dengan', 'dgn': 'dengan', 'sm': 'dengan',
    'dr': 'dari',
    'utk': 'untuk', 'tuk': 'untuk', 'u': 'untuk',
    'udh': 'sudah', 'sdh': 'sudah', 'udah': 'sudah', 'uda': 'sudah',
    'blm': 'belum', 'blom': 'belum',
    'lg': 'lagi', 'lgi': 'lagi',
    'skrg': 'sekarang', 'skrng': 'sekarang', 'skg': 'sekarang',
    'trs': 'terus', 'trus': 'terus',
    'bs': 'bisa', 'bsa': 'bisa',
    'hrs': 'harus',
    'jd': 'jadi', 'jdi': 'jadi',
    'dpt': 'dapat',
    'pke': 'pakai', 'pake': 'pakai', 'pk': 'pakai', 'makek' : 'pakai',
    'gmn': 'bagaimana', 'gimana': 'bagaimana',
    'knp': 'kenapa', 'knapa': 'kenapa',
    'dmn': 'dimana', 'dmna': 'dimana',
    'bgt': 'banget', 'bngt': 'banget', 'bngtt': 'banget',
    'bgtt': 'banget', 'bangett': 'banget',
    'mantap': 'bagus', 'mantab': 'bagus', 'mantapp': 'bagus',
    'keren': 'bagus', 'top': 'bagus',
    'jelek': 'buruk', 'parah': 'buruk', 'ancur': 'buruk', 'sampah': 'buruk',
    'apk': 'aplikasi', 'app': 'aplikasi',
    'eror': 'error', 'erorr': 'error', 'errorr': 'error',
    'lemot': 'lambat', 'lelet': 'lambat',
    'lag': 'lambat', 'ngelag': 'lambat',
    'crash': 'error', 'force close': 'error',
    'login': 'masuk', 'log in': 'masuk',
    'logout': 'keluar',
    'update': 'perbarui', 'upd': 'perbarui',
    'verif': 'verifikasi', 'verifikasiin': 'verifikasi',
    'makasih': 'terima kasih', 'makasi': 'terima kasih',
    'thx': 'terima kasih', 'thanks': 'terima kasih', 'tq': 'terima kasih',
    'org': 'orang', 'ornag': 'orang',
    'msh': 'masih',
    'bkn': 'bukan',
    'gpp': 'tidak apa apa', 'gapapa': 'tidak apa apa',
    'dlu': 'dulu', 'dl': 'dulu',
    'ntar': 'nanti', 'tar': 'nanti',
    'kmrn': 'kemarin', 'kmarin': 'kemarin',
    'mnggu': 'minggu', 'dlm' : 'dalam', 'bener' : 'benar',
}

print(f"Slang dictionary size: {len(idslang_dict)} entries")

Slang dictionary size: 114 entries


In [9]:
def normalize_slang(text):
    words = text.split()
    normalized = [idslang_dict.get(word, word) for word in words]
    return ' '.join(normalized)

df_bluRev['cleaned_content'] = df_bluRev['cleaned_content'].apply(normalize_slang)

changed_mask = (
    df_bluRev['content'].astype(str) != df_bluRev['cleaned_content']
)

df_bluRev.loc[changed_mask, ['content', 'cleaned_content']].head(10)

,content,cleaned_content
0,"verifikasi wajah tidak jelas,stuck di pastikan...",verifikasi wajah tidak jelasstuck di pastikan ...
1,hihihihi kog ngeri nian baca review penggunany...,hihihihi kog ngeri nian baca review penggunany...
2,gak perlu di instal hanya menghabiskan paket d...,tidak perlu di instal hanya menghabiskan paket...
3,wish keren sih..ini bank digital yang aku suka..,wish bagus sihini bank digital yang saya suka
4,"aplikasi kocak, verifikasi wajah gak bisa bisa...",aplikasi kocak verifikasi wajah tidak bisa bis...
5,"mantap, pembayarannya penarikan nya aman, semo...",bagus pembayarannya penarikan nya aman semoga ...
6,mantap.. tingkatkan terus kualitasnya,bagus tingkatkan terus kualitasnya
7,sampe pegel selfie.. pastikan smartphone tidak...,sampe pegel selfie pastikan smartphone tidak b...
9,"Kyc lupa bawa KK ke metmall jadi makan sushi, ...",kyc lupa bawa kk ke metmall jadi makan sushi a...
10,"Saya sangat senang sekali pakai Blu BCA,Transa...",saya sangat senang sekali pakai blu bcatransak...


### **Tokenization**


In [10]:
nltk.download('punkt')
nltk.download('punkt_tab')

df_bluRev['tokens'] = df_bluRev['cleaned_content'].apply(word_tokenize)
df_bluRev['token_count'] = df_bluRev['tokens'].apply(len)
df_bluRev[['cleaned_content', 'tokens', 'token_count']].head(5)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,cleaned_content,tokens,token_count
0,verifikasi wajah tidak jelasstuck di pastikan ...,"[verifikasi, wajah, tidak, jelasstuck, di, pas...",19
1,hihihihi kog ngeri nian baca review penggunany...,"[hihihihi, kog, ngeri, nian, baca, review, pen...",18
2,tidak perlu di instal hanya menghabiskan paket...,"[tidak, perlu, di, instal, hanya, menghabiskan...",10
3,wish bagus sihini bank digital yang saya suka,"[wish, bagus, sihini, bank, digital, yang, say...",8
4,aplikasi kocak verifikasi wajah tidak bisa bis...,"[aplikasi, kocak, verifikasi, wajah, tidak, bi...",11


In [11]:
total_tokens = df_bluRev['tokens'].apply(len).sum()
print(f"Total tokens in dataset: {total_tokens}")

Total tokens in dataset: 321483


### **Stopwords Removal**

In [12]:
try:
    stopwords.words('indonesian')
except LookupError:
    nltk.download('stopwords')

stop_words = set(
    stopwords.words('indonesian') +
    ["nya", "aja", "ya", "aplikasi", "bank", "blu", "bca", "digital"]
)

print(f"Total stopwords: {len(stop_words)}")
print("Sample stopwords:", list(stop_words)[:20])

Total stopwords: 765
Sample stopwords: ['gunakan', 'ibu', 'sesaat', 'karenanya', 'hari', 'seluruh', 'untuk', 'ibaratnya', 'awal', 'keinginan', 'semampunya', 'beginian', 'dipertanyakan', 'ditegaskan', 'sebagaimana', 'menyampaikan', 'semata', 'bapak', 'maupun', 'seperlunya']


In [13]:
df_bluRev['filtered_tokens'] = df_bluRev['tokens'].apply(
    lambda tokens: [word for word in tokens if word not in stop_words]
)
df_bluRev['final_token_count'] = df_bluRev['filtered_tokens'].apply(len)
df_bluRev[['tokens', 'filtered_tokens', 'final_token_count']].head(10)

,tokens,filtered_tokens,final_token_count
0,"[verifikasi, wajah, tidak, jelasstuck, di, pas...","[verifikasi, wajah, jelasstuck, pastikan, hp, ...",10
1,"[hihihihi, kog, ngeri, nian, baca, review, pen...","[hihihihi, kog, ngeri, nian, baca, review, pen...",14
2,"[tidak, perlu, di, instal, hanya, menghabiskan...","[instal, menghabiskan, paket, data, cuk]",5
3,"[wish, bagus, sihini, bank, digital, yang, say...","[wish, bagus, sihini, suka]",4
4,"[aplikasi, kocak, verifikasi, wajah, tidak, bi...","[kocak, verifikasi, wajah, terblokir, lawak]",5
5,"[bagus, pembayarannya, penarikan, nya, aman, s...","[bagus, pembayarannya, penarikan, aman, semoga...",8
6,"[bagus, tingkatkan, terus, kualitasnya]","[bagus, tingkatkan, kualitasnya]",3
7,"[sampe, pegel, selfie, pastikan, smartphone, t...","[sampe, pegel, selfie, pastikan, smartphone, b...",10
8,"[sangat, membantu]",[membantu],1
9,"[kyc, lupa, bawa, kk, ke, metmall, jadi, makan...","[kyc, lupa, bawa, kk, metmall, makan, sushi, b...",32


### **Stemming**

In [ ]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stem_tokens(tokens):
    return [stemmer.stem(token) for token in tokens]

df_bluRev['tokens_stemmed'] = df_bluRev['filtered_tokens'].apply(stem_tokens)
df_bluRev[['filtered_tokens', 'tokens_stemmed']].head(10)

In [ ]:
def normalize_repeated_chars(tokens):
    # mengubah huruf berulang jadi 1
    return re.sub(r'(.)\1+', r'\1', tokens)

df_bluRev['tokens_stemmed'] = df_bluRev['tokens_stemmed'].apply(
    lambda tokens: [normalize_repeated_chars(token) for token in tokens])

### **Recombine into sentences**

In [ ]:
df_bluRev['final_content'] = df_bluRev['tokens_stemmed'].apply(lambda x: ' '.join(x))
df_bluRev[['content', 'final_content']].head(10)

,content,final_content
0,"verifikasi wajah tidak jelas,stuck di pastikan...",verifikasi wajah jelastuck pasti hp gerak hp t...
1,hihihihi kog ngeri nian baca review penggunany...,hihihihi kog ngeri nian baca review guna horor...
2,gak perlu di instal hanya menghabiskan paket d...,instal habis paket data cuk
3,wish keren sih..ini bank digital yang aku suka..,wish bagus sihini suka
4,"aplikasi kocak, verifikasi wajah gak bisa bisa...",kocak verifikasi wajah blokir lawak
5,"mantap, pembayarannya penarikan nya aman, semo...",bagus bayar tari aman moga aman amanah amin
6,mantap.. tingkatkan terus kualitasnya,bagus tingkat kualitas
7,sampe pegel selfie.. pastikan smartphone tidak...,sampe gel selfie pasti smartphone goyang kalo ...
8,sangat membantu,bantu
9,"Kyc lupa bawa KK ke metmall jadi makan sushi, ...",kyc lupa bawa k metmal makan sushi bagus anak ...


### **Processed Data Validation**

In [ ]:
# Compare before vs after
print("Before vs After")
display(df_bluRev[['content', 'final_content']].sample(5))

Before vs After


,content,final_content
21031,Bagus aplikasinya....sesuai axpektasi...👍,bagus aplikasinyasesuai axpektasi
14075,Sanggat bermanfaat,sangat manfat
22523,"Numpang nama besar BCA doang , dasar prodak ga...",numpang nama doang dasar prodak
1787,aplikasi sangat ribet dan proses nya sangat la...,ribet proses nasabah tungu hari pe rgantian no...
26380,Sipllahhh,siplah


In [ ]:
empty_text = (df_bluRev['final_content'].str.strip() == '').sum()
print(f"\nEmpty text after preprocessing: {empty_text}")


Empty text after preprocessing: 373


In [ ]:
df_bluRev = df_bluRev[df_bluRev['final_content'].str.strip() != '']
empty_text = (df_bluRev['final_content'].str.strip() == '').sum()
print(f"Empty text after removal: {empty_text}")

Empty text after removal: 0


In [ ]:
# Token length statistics
df_bluRev['final_token_count'] = df_bluRev['tokens_stemmed'].apply(len)

print("\n=== Token Statistics ===")
print(f"Average tokens: {df_bluRev['final_token_count'].mean():.2f}")
print(f"Max tokens: {df_bluRev['final_token_count'].max()}")
print(f"Min tokens: {df_bluRev['final_token_count'].min()}")


=== Token Statistics ===
Average tokens: 6.66
Max tokens: 66
Min tokens: 1


In [ ]:
# Check sample of very short texts
print("\n=== Very Short Texts (<=2 tokens) ===")
display(df_bluRev[df_bluRev['final_token_count'] <= 2][['final_content']].head(5))


=== Very Short Texts (<=2 tokens) ===


,final_content
8,bantu
22,bagus bagus
24,bagus
25,amanah
27,bagus pakai


In [ ]:
# Check sample of very long texts
print("\n=== Very Long Texts (Top 5) ===")
display(df_bluRev.sort_values(by='final_token_count', ascending=False)[['final_content']].head(5))


=== Very Long Texts (Top 5) ===


,final_content
23424,tdinya mls antri bikim atm bri cba dftrin paka...
3009,slmt sore maf btlkn sj pinjeman niat pinjam br...
13302,bayar pakai qr tgl april proses gagal saldo ku...
650,astagfirulahaladzim sgt uji sabar puluh kali c...
15904,halo ka aj tdi download d sruh mba cs ny downl...


### **EDA**

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

print("=== Statistik Token Setelah Preprocessing ===")
mean_raw   = df_bluRev['token_count'].mean()
mean_stem  = df_bluRev['final_token_count'].mean()
print(f"  Rata-rata token (sebelum stopword removal) : {mean_raw:.1f}")
print(f"  Rata-rata token (sesudah stemming)          : {mean_stem:.1f}  (reduksi: {(1-mean_stem/mean_raw)*100:.1f}%)")
print(f"  Median token (sesudah stemming)             : {df_bluRev['final_token_count'].median():.1f}")
print(f"  Min token (sesudah stemming)                : {df_bluRev['final_token_count'].min()}")
print(f"  Max token (sesudah stemming)                : {df_bluRev['final_token_count'].max()}")

print("\n=== Token Setelah Stemming per Platform ===")
print(df_bluRev['final_token_count']
      .agg(['mean','median','min','max'])
      .round(1)
      .rename(index={'mean':'Rata-rata','median':'Median','min':'Min','max':'Max'})
      .to_string())

all_raw  = [t for toks in df_bluRev['tokens'].tolist()         for t in toks]
all_stem = [t for toks in df_bluRev['tokens_stemmed'].tolist() for t in toks]
print(f"\n=== Statistik Kosakata ===")
print(f"  Token unik (sebelum preprocessing)  : {len(set(all_raw)):,}")
print(f"  Token unik (setelah stemming)        : {len(set(all_stem)):,}")
print(f"  Reduksi kosakata                     : {(1-len(set(all_stem))/len(set(all_raw)))*100:.1f}%")

=== Statistik Token Setelah Preprocessing ===
  Rata-rata token (sebelum stopword removal) : 12.3
  Rata-rata token (sesudah stemming)          : 6.7  (reduksi: 45.7%)
  Median token (sesudah stemming)             : 4.0
  Min token (sesudah stemming)                : 1
  Max token (sesudah stemming)                : 66

=== Token Setelah Stemming per Platform ===
Rata-rata     6.7
Median        4.0
Min           1.0
Max          66.0

=== Statistik Kosakata ===
  Token unik (sebelum preprocessing)  : 18,364
  Token unik (setelah stemming)        : 13,693
  Reduksi kosakata                     : 25.4%


In [ ]:
ORDER = ['BluApp Review']
COLORS = ['#2E86AB','#A23B72','#F18F01','#C73E1D','#3B1F2B']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(len(ORDER))

ax = axes[0]
w = 0.35
before_m = [df_bluRev['token_count'].mean() for n in ORDER]
after_m  = [df_bluRev['final_token_count'].mean() for n in ORDER]
b1 = ax.bar(x - w/2, before_m, w, label='Sebelum Preprocessing', color='#2E86AB', alpha=0.85)
b2 = ax.bar(x + w/2, after_m,  w, label='Setelah Preprocessing',  color='#F18F01', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(ORDER, fontsize=11)
ax.set_ylabel('Rata-rata Token', fontsize=11)
ax.set_title('Rata-rata Token per Platform:\nSebelum vs Sesudah Preprocessing', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
for bar in b1: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+3, f'{bar.get_height():.0f}', ha='center', fontsize=9)
for bar in b2: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+3, f'{bar.get_height():.0f}', ha='center', fontsize=9)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

ax = axes[1]
raw_m     = [df_bluRev['token_count'].mean()]
clean_m   = [df_bluRev['final_token_count'].mean()]
removed_m = [r - c for r, c in zip(raw_m, clean_m)]
b1 = ax.bar(x, clean_m,   0.55, label='Token Dipertahankan',               color='#2E86AB', alpha=0.9)
b2 = ax.bar(x, removed_m, 0.55, bottom=clean_m, label='Token Dihapus',      color='#E87040', alpha=0.75)
ax.set_xticks(x); ax.set_xticklabels(ORDER, fontsize=11)
ax.set_ylabel('Rata-rata Token', fontsize=11)
ax.set_title('Komposisi Token per Platform:\nDipertahankan vs Dihapus', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
for i, (c, r) in enumerate(zip(clean_m, raw_m)):
    ax.text(i, r+3, f'{c/r*100:.0f}%\ndipertahankan', ha='center', fontsize=8)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.suptitle('EDA Sesudah Preprocessing', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('visualisasi/eda_after_preprocessing.png', dpi=150, bbox_inches='tight')
plt.show()

freq  = Counter(all_stem)
top8 = freq.most_common(8)
words, counts = zip(*top8)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(list(reversed(words)), list(reversed(counts)), color='#2E86AB', alpha=0.85)
for i, v in enumerate(reversed(counts)):
    ax.text(v+30, i, f'{v:,}', va='center', fontsize=9)
ax.set_xlabel('Frekuensi', fontsize=12)
ax.set_title('Top 8 Kata Paling Sering Muncul Setelah Preprocessing (Stemmed)',
             fontsize=12, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('visualisasi/eda_top8_words.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ EDA Sesudah Preprocessing selesai.")

NameError: name 'plt' is not defined

In [ ]:
df_bluRev.loc[:,["content","score","thumbsUpCount", "reviewCreatedVersion", "at", "replyContent", "repliedAt", "tokens", "token_count", "filtered_tokens", "final_token_count", "tokens_stemmed", "final_content"]]

,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,tokens,token_count,filtered_tokens,final_token_count,tokens_stemmed,final_content
0,"verifikasi wajah tidak jelas,stuck di pastikan...",1,0,NaN,2026-05-16 23:08:41,"Hai, Kak Darma. Maaf atas ketidaknyamanannya. ...",2026-05-17 00:07:20,"[verifikasi, wajah, tidak, jelasstuck, di, pas...",19,"[verifikasi, wajah, jelasstuck, pastikan, hp, ...",10,"[verifikasi, wajah, jelastuck, pasti, hp, gera...",verifikasi wajah jelastuck pasti hp gerak hp t...
1,hihihihi kog ngeri nian baca review penggunany...,1,0,NaN,2026-05-16 22:10:32,"Hai, Kak Maulana. Maaf atas ketidaknyamanannya...",2026-05-16 23:06:54,"[hihihihi, kog, ngeri, nian, baca, review, pen...",18,"[hihihihi, kog, ngeri, nian, baca, review, pen...",14,"[hihihihi, kog, ngeri, nian, baca, review, gun...",hihihihi kog ngeri nian baca review guna horor...
2,gak perlu di instal hanya menghabiskan paket d...,1,0,2.12.10,2026-05-16 21:52:46,"Hai, Kak Jaki. Maaf atas ketidaknyamanannya. K...",2026-05-16 21:55:55,"[tidak, perlu, di, instal, hanya, menghabiskan...",10,"[instal, menghabiskan, paket, data, cuk]",5,"[instal, habis, paket, data, cuk]",instal habis paket data cuk
3,wish keren sih..ini bank digital yang aku suka..,5,0,2.12.10,2026-05-16 21:11:37,"Hai, Kak Alan. Terima kasih atas ulasannya. Se...",2026-05-16 21:54:10,"[wish, bagus, sihini, bank, digital, yang, say...",8,"[wish, bagus, sihini, suka]",4,"[wish, bagus, sihini, suka]",wish bagus sihini suka
4,"aplikasi kocak, verifikasi wajah gak bisa bisa...",1,0,NaN,2026-05-16 20:47:26,"Hai, Kak Muhammad. Maaf atas ketidaknyamananny...",2026-05-16 21:53:27,"[aplikasi, kocak, verifikasi, wajah, tidak, bi...",11,"[kocak, verifikasi, wajah, terblokir, lawak]",5,"[kocak, verifikasi, wajah, blokir, lawak]",kocak verifikasi wajah blokir lawak
...,...,...,...,...,...,...,...,...,...,...,...,...,...
26525,"Selamat pagi masyarakat BCA DIGITAL, hehehehe😁...",5,0,1.3.2,2021-07-02 09:59:18,"Hai, Kak Ridwan Mukti Sastama. Terima kasih at...",2021-07-04 15:24:53,"[selamat, pagi, masyarakat, bca, digital, hehe...",6,"[selamat, pagi, masyarakat, hehehehe]",4,"[selamat, pagi, masyarakat, hehehehe]",selamat pagi masyarakat hehehehe
26526,"Ribet ...minta diaktifkan screen lock, dah akt...",1,0,1.3.2,2021-07-02 09:53:23,"Hai, Kak Hidayat Hidayat. Terima kasih untuk f...",2021-07-04 14:59:53,"[ribet, minta, diaktifkan, screen, lock, dah, ...",11,"[ribet, diaktifkan, screen, lock, dah, aktif, ...",7,"[ribet, aktif, scren, lock, dah, aktif, jalan]",ribet aktif scren lock dah aktif jalan
26527,Video call kok ngk berhasil,5,0,1.3.2,2021-07-02 09:50:42,"Hai, Kak Roni Susanto. Terima kasih untuk feed...",2021-07-04 14:49:57,"[video, call, kok, tidak, berhasil]",5,"[video, call, berhasil]",3,"[video, cal, hasil]",video cal hasil
26528,Pertamax....test,3,1,NaN,2021-07-02 09:42:19,"Hai, Kak Zainal CorRosion. Terima kasih atas u...",2021-07-04 14:34:47,[pertamaxtest],1,[pertamaxtest],1,[pertamaxtest],pertamaxtest


In [ ]:
selected_col = ['content', 'score', 'at', 'thumbsUpCount', 'replyContent', 'final_content', 'tokens_stemmed']

df_bluRev_preprocessed = df_bluRev[selected_col].copy()
df_bluRev_preprocessed.to_csv(r'../data/bluapp_reviews_preprocessed.csv', index = False)